# 🌧 Krea-2 LoRA — Rainy Window Background Generator
### For Ashlee's Daily Planner

**Before running anything:**
1. Make sure the runtime has a GPU: `Runtime → Change runtime type → T4 GPU`
2. You'll need a Hugging Face token (read-access) — paste it in Cell 2
3. Run each cell top-to-bottom by clicking the ▶ button on the left of each cell

Each cell takes ~1–2 minutes. The final image downloads automatically.

In [ ]:
# ── CELL 1: Install dependencies (takes ~60s) ──────────────────────────────
print('Installing libraries...')
!pip install -q diffusers transformers accelerate safetensors huggingface_hub Pillow torch torchvision
print('✓ Done')

In [ ]:
# ── CELL 2: Paste your Hugging Face token here ─────────────────────────────
# Get it from: huggingface.co → Settings → Access Tokens → New token (read)

HF_TOKEN = "hf_PASTE_YOUR_TOKEN_HERE"   # ← replace this

from huggingface_hub import login
login(token=HF_TOKEN, add_to_git_credential=False)
print('✓ Logged in to Hugging Face')

In [ ]:
# ── CELL 3: Load the Krea-2 base model + LoRA weights (~5–8 min first run) ─
import torch
from diffusers import FluxPipeline
from huggingface_hub import hf_hub_download

print('Loading Krea-2 base model (this may take a few minutes)...')

pipe = FluxPipeline.from_pretrained(
    "krea/Krea-2",
    torch_dtype=torch.bfloat16,
    token=HF_TOKEN
)
pipe = pipe.to("cuda")

print('Loading rainy window LoRA weights...')
pipe.load_lora_weights(
    "krea/Krea-2-LoRA-rainywindow",
    token=HF_TOKEN
)

print('✓ Model ready — GPU VRAM used:', round(torch.cuda.memory_allocated()/1e9, 1), 'GB')

In [ ]:
# ── CELL 4: Set your prompt and generate ──────────────────────────────────
# Edit the prompt below, then run this cell.
# The trigger phrase "rainy window style" must be included — it activates the LoRA.

PROMPT = (
    "rainy window style, "
    "blurred greenery and forest outside, muted earthy tones, "
    "photorealistic, ultra-detailed water droplets on glass, "
    "soft bokeh, cinematic, overcast natural light, 8K"
)

# Image size — 1344×768 is 16:9 widescreen, perfect for a desktop background
WIDTH  = 1344
HEIGHT = 768
STEPS  = 30   # more steps = more detail but slower (20–40 is good)
SEED   = 42   # change this number for a different result

print('Generating... (~30–60s)')
generator = torch.Generator("cuda").manual_seed(SEED)

result = pipe(
    prompt=PROMPT,
    width=WIDTH,
    height=HEIGHT,
    num_inference_steps=STEPS,
    guidance_scale=3.5,
    generator=generator
)

image = result.images[0]

# Preview in the notebook
from IPython.display import display
display(image)
print('✓ Generated')

In [ ]:
# ── CELL 5: Save and download ──────────────────────────────────────────────
import os
from google.colab import files

filename = f"rainy_window_seed{SEED}.jpg"
image.save(filename, quality=97)
print(f'Saved as {filename} ({os.path.getsize(filename)//1024} KB)')

files.download(filename)  # triggers browser download automatically
print('✓ Download started!')

In [ ]:
# ── CELL 6 (optional): Try more variations ─────────────────────────────────
# Change SEED or PROMPT and re-run cells 4 + 5 to generate more options.
# Some seed ideas to try:

VARIATIONS = [
    {"seed": 101, "note": "city lights at night",
     "extra": "city lights bokeh at night, neon reflections on wet glass"},
    {"seed": 202, "note": "misty mountains",
     "extra": "misty mountain valley, fog, minimal, zen"},
    {"seed": 303, "note": "abstract warm tones",
     "extra": "abstract warm amber and ochre background, minimal"},
]

BASE = "rainy window style, water droplets on glass, photorealistic, 8K, "

for v in VARIATIONS:
    print(f"\n🎲 Seed {v['seed']}: {v['note']}")
    gen = torch.Generator("cuda").manual_seed(v["seed"])
    img = pipe(
        prompt=BASE + v["extra"],
        width=WIDTH, height=HEIGHT,
        num_inference_steps=STEPS,
        guidance_scale=3.5,
        generator=gen
    ).images[0]
    fname = f"rainy_window_seed{v['seed']}.jpg"
    img.save(fname, quality=97)
    display(img)
    files.download(fname)

print('✓ All variations done')

## ✅ What to do with the downloaded image

You now have a `.jpg` file. Two options:

**Option A — Drag into Settings (easiest)**  
Open your daily planner → Settings → AI Background → click the upload icon → choose your file. The planner will use it as the background with your animated rain drops on top.

**Option B — Host it yourself**  
Upload to any image host (Imgur, Cloudinary, your own GitHub, etc.) and paste the URL into the planner's AI Background URL field.

The animated canvas drops continue running on top regardless — you get photorealistic rain *behind* the glass AND physics-based rain beads *on* the glass at the same time.